# RSNA Pneumonia Detection: Step 2 - RT-DETR-L 72-Epoch Chunked Training & Proposal Harvesting

> **Project:** Foundation-Model-Based ROI Refinement of RT-DETR Predictions for False-Positive Reduction  
> **Architecture:** Primary Detector (RT-DETR-L, 46.2M parameters)  
> **Research Target:** 72 Epochs baseline from literature (AdamW, $\text{lr}=1\times 10^{-4}$, effective batch size 16)  
> **Kaggle 9-Hour Strategy:** 10-Epoch chunked sessions with automated `last.pt` checkpoint resumption.

---

### Target Baseline Metrics from Published Benchmark:
| Metric | Paper RT-DETR-L Target |
| :--- | :--- |
| **mAP@0.5** | **78.5%** |
| **mAP@0.5:0.95** | **48.1%** |
| **Recall@0.5** | **80.2%** |
| **Precision@0.5** | **76.6%** |
| **Parameters** | **46.2M** |
| **Baseline FP/image** | **~0.75 - 0.78** |


In [1]:
# Cell 1: Install required libraries
!pip install -q ultralytics pydicom opencv-python pandas numpy scikit-learn


In [2]:
# Cell 2: Imports & GPU Verification
import os
import sys
import glob
import json
import shutil
import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import RTDETR

print(f"PyTorch Version  : {torch.__version__}")
print(f"CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count     : {torch.cuda.device_count()}")
    print(f"Active GPU       : {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM Total   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: CUDA is not available! A GPU accelerator (T4 x2 or P100) is required.")


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch Version  : 2.11.0+cu128
CUDA Available   : True
Device Count     : 2
Active GPU       : Tesla T4
GPU VRAM Total   : 15.64 GB


## 1. Automated Dataset Discovery

Locates `rsna_pneumonia.yaml` from either:
1. Mounted Kaggle Dataset (e.g. `/kaggle/input/rsna-rtdetr-dataset/...`)
2. Working directory from Step 1 (`/kaggle/working/rsna_rtdetr_dataset/...`)


In [3]:
# Cell 3: Locate Dataset Root & Generate Dynamic Runtime YAML
# 1. If only zip is present in /kaggle/input, auto-unzip it to /kaggle/working/
if not os.path.exists("/kaggle/working/rsna_rtdetr_dataset/images/train"):
    zip_candidates = glob.glob("/kaggle/input/**/rsna_rtdetr_dataset.zip", recursive=True)
    if zip_candidates:
        print(f"[UNZIP] Found dataset zip archive at: {zip_candidates[0]}")
        os.makedirs("/kaggle/working/rsna_rtdetr_dataset", exist_ok=True)
        shutil.unpack_archive(zip_candidates[0], "/kaggle/working/rsna_rtdetr_dataset")
        print("[UNZIP] Successfully extracted to /kaggle/working/rsna_rtdetr_dataset")

# 2. Locate the folder containing images/train
def locate_dataset_root():
    candidates = [
        "/kaggle/working/rsna_rtdetr_dataset",
        "/kaggle/input/notebooks/kota2006/notebook5e79297f62/rsna_rtdetr_dataset",
        "/kaggle/input/rsna-rtdetr-dataset/rsna_rtdetr_dataset",
        "/kaggle/input/rsna-rtdetr-dataset",
        "/kaggle/input/rsna-rtdetr-preprocessed/rsna_rtdetr_dataset",
        "/kaggle/input/rsna-rtdetr-preprocessed"
    ]
    for p in candidates:
        if os.path.exists(os.path.join(p, "images", "train")):
            return os.path.normpath(os.path.abspath(p))
            
    # Recursive search if in an unexpected subfolder
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "images" in dirs:
            train_sub = os.path.join(root, "images", "train")
            if os.path.exists(train_sub) and len(os.listdir(train_sub)) > 0:
                return os.path.normpath(os.path.abspath(root))

    raise FileNotFoundError("Could not find preprocessed dataset folder containing images/train!")

DATASET_ROOT = locate_dataset_root()
print(f"[FOUND] Preprocessed Dataset Root: {DATASET_ROOT}")

# 3. Write a fresh runtime YAML pointing directly to the discovered location
RUNTIME_YAML = "/kaggle/working/rsna_pneumonia_runtime.yaml"
runtime_yaml_content = f"""path: {DATASET_ROOT}
train: images/train
val: images/val
test: images/test

names:
  0: pneumonia
"""
with open(RUNTIME_YAML, "w") as f:
    f.write(runtime_yaml_content)

print(f"[CONFIG] Generated Runtime YAML: {RUNTIME_YAML}")

TEST_IMAGES_DIR = os.path.join(DATASET_ROOT, "images", "test")
TEST_LABELS_DIR = os.path.join(DATASET_ROOT, "labels", "test")

[UNZIP] Found dataset zip archive at: /kaggle/input/notebooks/kota2006/notebook5e79297f62/rsna_rtdetr_dataset.zip
[UNZIP] Successfully extracted to /kaggle/working/rsna_rtdetr_dataset
[FOUND] Preprocessed Dataset Root: /kaggle/working/rsna_rtdetr_dataset
[CONFIG] Generated Runtime YAML: /kaggle/working/rsna_pneumonia_runtime.yaml


## 2. Multi-Session Checkpoint Detection & Chunk Scheduler

To fit 72 epochs within Kaggle's 9-hour limit, training runs in **10-epoch increments**:
- **Session 1:** Epochs 1 – 10
- **Session 2:** Epochs 11 – 20
- ...
- **Session 8:** Epochs 71 – 72

The cell below checks for previous checkpoints (`last.pt`), determines the current completed epoch, and automatically sets the next chunk target.


In [4]:
# Cell 4: Checkpoint Discovery & Target Epoch Calculation
PROJECT_DIR = "/kaggle/working/experiments/baseline_rtdetr"
RUN_NAME = "train_run"
EXP_DIR = os.path.join(PROJECT_DIR, RUN_NAME)
WEIGHTS_DIR = os.path.join(EXP_DIR, "weights")

# 0. Automatically clean up if previous attempt in /kaggle/working failed with NaNs
working_results = os.path.join(EXP_DIR, "results.csv")
working_last = os.path.join(WEIGHTS_DIR, "last.pt")
needs_reset = False

if os.path.exists(working_results):
    try:
        df_chk = pd.read_csv(working_results)
        if df_chk.isna().any().any():
            print("[RESET] Detected NaN loss in working results.csv! Resetting corrupted run...")
            needs_reset = True
    except Exception:
        pass

if os.path.exists(working_last) and not needs_reset:
    try:
        ckpt_test = torch.load(working_last, map_location="cpu", weights_only=False)
        mdl = ckpt_test.get("model", None)
        if mdl is not None:
            s = mdl.state_dict() if hasattr(mdl, "state_dict") else mdl
            for k, v in list(s.items())[:20]:
                if torch.is_tensor(v) and (torch.isnan(v).any() or torch.isinf(v).any()):
                    print(f"[RESET] Detected NaN in model tensor '{k}'! Resetting corrupted working run...")
                    needs_reset = True
                    break
    except Exception:
        pass

if needs_reset:
    shutil.rmtree(EXP_DIR, ignore_errors=True)
    print("[RESET] Corrupted working run cleared. Ready to resume from clean checkpoint in /kaggle/input/.")

CHUNK_SIZE = 10
TOTAL_TARGET_EPOCHS = 72

def discover_checkpoint():
    """Checks attached datasets and working directory for existing checkpoints and determines completed epochs."""
    candidates = [
        os.path.join(WEIGHTS_DIR, "last.pt"),
        "/kaggle/input/rtdetr-chunk-checkpoint/last.pt",
        "/kaggle/input/rtdetr-checkpoints/last.pt",
        "/kaggle/input/rtdetr-baseline-checkpoint/last.pt"
    ]
    input_lasts = glob.glob("/kaggle/input/**/last.pt", recursive=True)
    candidates.extend(input_lasts)

    for p in candidates:
        if os.path.exists(p):
            try:
                try:
                    ckpt = torch.load(p, map_location="cpu", weights_only=False)
                except TypeError:
                    ckpt = torch.load(p, map_location="cpu")

                # Method 1: Check results.csv in parent folder
                parent_dir = os.path.dirname(os.path.dirname(p))
                results_csv = os.path.join(parent_dir, "results.csv")
                if os.path.exists(results_csv):
                    try:
                        df_res = pd.read_csv(results_csv)
                        if len(df_res) > 0 and not df_res.isna().any().any():
                            completed = len(df_res)
                            print(f"[CHECKPOINT] Found {p}")
                            print(f"[CHECKPOINT] Verified {completed} epochs completed via results.csv")
                            return p, completed
                    except Exception:
                        pass

                # Method 2: Check train_args['epochs'] if finalized (where ckpt['epoch'] == -1)
                train_args = ckpt.get("train_args", {})
                if isinstance(train_args, dict) and "epochs" in train_args:
                    completed = int(train_args["epochs"])
                    print(f"[CHECKPOINT] Found {p}")
                    print(f"[CHECKPOINT] Verified {completed} epochs completed via train_args")
                    return p, completed

                # Method 3: Check raw epoch
                raw_ep = ckpt.get("epoch", -1)
                if raw_ep != -1:
                    completed = raw_ep + 1
                    print(f"[CHECKPOINT] Found {p} (epoch {completed})")
                    return p, completed

                # Default fallback
                print(f"[CHECKPOINT] Found checkpoint {p}. Assuming 10 completed epochs from Session 1.")
                return p, 10

            except Exception as e:
                print(f"Could not inspect {p}: {e}")

    return None, 0

CHECKPOINT_PATH, COMPLETED_EPOCHS = discover_checkpoint()

REMAINING_EPOCHS = TOTAL_TARGET_EPOCHS - COMPLETED_EPOCHS
THIS_SESSION_EPOCHS = min(CHUNK_SIZE, REMAINING_EPOCHS)
TARGET_TOTAL_EPOCHS = COMPLETED_EPOCHS + THIS_SESSION_EPOCHS

print(f"\n{'='*65}")
print(f"CHUNK TRAINING STATUS:")
print(f"  Completed Epochs So Far  : {COMPLETED_EPOCHS}/{TOTAL_TARGET_EPOCHS}")
print(f"  Training In This Session : {THIS_SESSION_EPOCHS} epochs")
print(f"  Total Progress After Run : {TARGET_TOTAL_EPOCHS}/{TOTAL_TARGET_EPOCHS} epochs")
print(f"{'='*65}\n")

[CHECKPOINT] Found /kaggle/input/datasets/kota2006/rtdetr-epoch10checkpoint/experiments/baseline_rtdetr/train_run/weights/last.pt
[CHECKPOINT] Verified 10 epochs completed via results.csv

CHUNK TRAINING STATUS:
  Completed Epochs So Far  : 10/72
  Training In This Session : 10 epochs
  Total Progress After Run : 20/72 epochs



## 3. Train RT-DETR-L with Paper Hyperparameters

We configure Ultralytics RT-DETR-L to match the published paper:
- **Architecture:** `rtdetr-l.pt` (46.2M parameters)
- **Optimizer:** `AdamW`
- **Initial Learning Rate:** $1\times 10^{-4}$ (`lr0=0.0001`, `lrf=0.01` cosine decay)
- **Batch Size:** 4 with `accumulate=4` $\rightarrow$ **Effective Batch Size = 16** (prevents 16GB T4 VRAM Out-of-Memory)
- **Medical Augmentations:** `mosaic=0.0`, `mixup=0.0`, `flipud=0.0` (preserves anatomical integrity)
- **Checkpointing:** `save_period=1` (saves weights every single epoch)


In [5]:
# Cell 5: Execute Chunked RT-DETR Training
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if COMPLETED_EPOCHS >= TOTAL_TARGET_EPOCHS:
    print(f"Goal of {TOTAL_TARGET_EPOCHS} epochs already reached! Proceeding to test evaluation.")
else:
    if CHECKPOINT_PATH is not None:
        print(f"[RESUME] Loading weights from previous chunk: {CHECKPOINT_PATH}")
        working_last = os.path.join(WEIGHTS_DIR, "last.pt")
        if os.path.abspath(CHECKPOINT_PATH) != os.path.abspath(working_last):
            shutil.copy(CHECKPOINT_PATH, working_last)
            CHECKPOINT_PATH = working_last
        model = RTDETR(CHECKPOINT_PATH)
    else:
        print("[FRESH] Starting from official pretrained RT-DETR-L weights (rtdetr-l.pt)...")
        model = RTDETR("rtdetr-l.pt")

    print(f"Starting training for {THIS_SESSION_EPOCHS} epochs in this session (FP32 Precision)...")

    results = model.train(
        data=RUNTIME_YAML,
        epochs=THIS_SESSION_EPOCHS,
        imgsz=640,
        batch=8,
        nbs=16,
        optimizer="AdamW",
        lr0=0.0001,
        weight_decay=0.0001,
        warmup_epochs=0.0 if COMPLETED_EPOCHS > 0 else 3.0,
        device=0 if torch.cuda.is_available() else "cpu",
        workers=4,
        project=PROJECT_DIR,
        name=RUN_NAME,
        exist_ok=True,
        save=True,
        save_period=1,
        amp=False,          # CRITICAL: FP32 eliminates attention and Hungarian matching NaN overflow
        plots=True,
        mosaic=0.0,
        mixup=0.0,
        flipud=0.0,
        fliplr=0.5,
        degrees=7.0,
        scale=0.10,
        hsv_v=0.15
    )

    print(f"\n[SUCCESS] Session complete! Model has now been trained for {TARGET_TOTAL_EPOCHS} total epochs.")

[RESUME] Loading weights from previous chunk: /kaggle/input/datasets/kota2006/rtdetr-epoch10checkpoint/experiments/baseline_rtdetr/train_run/weights/last.pt
Starting training for 10 epochs in this session (FP32 Precision)...
Ultralytics 8.4.185 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/rsna_pneumonia_runtime.yaml, degrees=7.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.15, imgsz

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/10      9.39G     0.4855      1.157     0.3799          5        640: 100% ━━━━━━━━━━━━ 2335/2335 1.2s/it 46:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 251/251 1.5it/s 2:52
                   all       4003       1421      0.218      0.305      0.174     0.0617

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/10        10G      0.472      1.155     0.3675          0        640: 100% ━━━━━━━━━━━━ 2335/2335 1.2s/it 46:14
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 251/251 1.5it/s 2:48
                   all       4003       1421      0.196      0.338      0.161     0.0595

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (11 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       9/10      9.91G       0.45     0.9941     0.3494          1        640: 100% ━━━━━━━━━━━━ 2335/2335 1.2s/it 46:18
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 251/251 1.5it/s 2:48
                   all       4003       1421      0.278      0.357      0.238     0.0964

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      10/10        10G     0.4559     0.9829     0.3533          2        640: 100% ━━━━━━━━━━━━ 2335/2335 1.2s/it 46:17
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 251/251 1.5it/s 2:48
                   all       4003       1421      0.286      0.355      0.243     0.0977

10 epochs completed in 8.196 hours.
Optimizer stripped from /kaggle/working/experiments/baseline_rtdetr/train_run/weights/last.pt, 66.2MB
Optimizer stripped from /kaggle/working/experiments/baseline_rtdetr/train_run/weights/best.pt, 66.2MB

Validating /kaggle/working/experiments/baseline_rtdetr/train_run/weights/best.pt...
Ultralytics 8.4.185 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rt-detr-l summary (fused): 315 layers, 31,985,795 parameters, 0 gradients, 105.3 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 251/251 1.8it/s 2:23
                   all       4003 

## 4. Package Checkpoint Archive for Next Session

Creates a zip archive of the checkpoint weights (`last.pt` and `best.pt`) and training logs.  
Save this archive as a Kaggle dataset so you can resume Session 2, 3, etc. seamlessly!


In [6]:
# Cell 6: Package Checkpoints for Easy Download / Next Session
checkpoint_zip = "/kaggle/working/rtdetr_chunk_checkpoint.zip"
checkpoint_folder = os.path.join(PROJECT_DIR, RUN_NAME)

if os.path.exists(checkpoint_folder):
    print("Creating compressed checkpoint archive...")
    shutil.make_archive(
        base_name="/kaggle/working/rtdetr_chunk_checkpoint",
        format="zip",
        root_dir=checkpoint_folder
    )
    print(f"[SUCCESS] Checkpoint saved at: {checkpoint_zip}")
    print("  -> Save this file as a Kaggle Dataset to easily resume the next 10 epochs in your next notebook!")


Creating compressed checkpoint archive...
[SUCCESS] Checkpoint saved at: /kaggle/working/rtdetr_chunk_checkpoint.zip
  -> Save this file as a Kaggle Dataset to easily resume the next 10 epochs in your next notebook!


## 5. Candidate Proposal Harvesting & Benchmark Metric Comparison

Once the detector has trained (or when benchmarking intermediate checkpoints):
1. Runs inference on the unseen held-out test split.
2. Extracts candidate bounding boxes with confidence scores.
3. Crops each candidate ROI with a **10% anatomical context margin** for Stage 2 (RAD-DINO).
4. Calculates **Precision, Recall, mAP@0.5, and False Positives per Image (FP/image)**.
5. Displays a comparison against the published paper target.


In [7]:
# Cell 7: Test Set Evaluation & 10% Context ROI Extraction
def compute_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter_w, inter_h = max(0.0, xB - xA), max(0.0, yB - yA)
    inter_area = inter_w * inter_h
    boxA_area = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxB_area = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])
    union_area = boxA_area + boxB_area - inter_area
    return inter_area / union_area if union_area > 0 else 0.0

def load_ground_truth(test_labels_dir, img_size=640):
    gt = {}
    if not os.path.exists(test_labels_dir):
        return gt
    for file in os.listdir(test_labels_dir):
        if not file.endswith(".txt"):
            continue
        pid = os.path.splitext(file)[0]
        boxes = []
        with open(os.path.join(test_labels_dir, file)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    _, xc, yc, w, h = map(float, parts)
                    x1 = (xc - w / 2.0) * img_size
                    y1 = (yc - h / 2.0) * img_size
                    x2 = (xc + w / 2.0) * img_size
                    y2 = (yc + h / 2.0) * img_size
                    boxes.append([x1, y1, x2, y2])
        gt[pid] = boxes
    return gt

best_model_path = os.path.join(WEIGHTS_DIR, "best.pt")
eval_model_path = best_model_path if os.path.exists(best_model_path) else os.path.join(WEIGHTS_DIR, "last.pt")

if os.path.exists(eval_model_path) and os.path.exists(TEST_IMAGES_DIR):
    print(f"\nEvaluating checkpoint on test split: {eval_model_path}")
    eval_model = RTDETR(eval_model_path)
    gt_dict = load_ground_truth(TEST_LABELS_DIR, img_size=640)
    test_files = [f for f in os.listdir(TEST_IMAGES_DIR) if f.lower().endswith(('.jpg', '.png'))]

    candidate_records = []
    total_tp = 0
    total_fp = 0
    total_gt = sum(len(b) for b in gt_dict.values())
    
    crops_dir = "/kaggle/working/candidate_rois/test"
    os.makedirs(crops_dir, exist_ok=True)

    print(f"Extracting candidate proposals from {len(test_files)} test images...")
    for idx, fname in enumerate(test_files):
        pid = os.path.splitext(fname)[0]
        img_path = os.path.join(TEST_IMAGES_DIR, fname)
        img = cv2.imread(img_path)
        if img is None:
            continue
        h_orig, w_orig = img.shape[:2]

        results = eval_model.predict(source=img_path, conf=0.15, imgsz=640, device=0 if torch.cuda.is_available() else "cpu", verbose=False)
        boxes_xyxy = results[0].boxes.xyxy.cpu().numpy()
        scores = results[0].boxes.conf.cpu().numpy()
        
        gt_boxes = gt_dict.get(pid, [])
        matched_gt = set()

        for b_idx, (box, score) in enumerate(zip(boxes_xyxy, scores)):
            x1, y1, x2, y2 = box
            w_box, h_box = x2 - x1, y2 - y1

            # Match with Ground Truth
            best_iou = 0.0
            best_gt_idx = -1
            for g_idx, g_box in enumerate(gt_boxes):
                if g_idx in matched_gt:
                    continue
                iou = compute_iou([x1, y1, x2, y2], g_box)
                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = g_idx

            if best_iou >= 0.40 and best_gt_idx != -1:
                is_tp = 1
                matched_gt.add(best_gt_idx)
                total_tp += 1
            else:
                is_tp = 0
                total_fp += 1

            # 10% Context-Padded Cropping for RAD-DINO
            pad_w = 0.10 * w_box
            pad_h = 0.10 * h_box
            crop_x1 = max(0, int(x1 - pad_w))
            crop_y1 = max(0, int(y1 - pad_h))
            crop_x2 = min(w_orig, int(x2 + pad_w))
            crop_y2 = min(h_orig, int(y2 + pad_h))
            
            crop_img = img[crop_y1:crop_y2, crop_x1:crop_x2]
            crop_fname = f"{pid}_crop_{b_idx}.jpg"
            crop_path = os.path.join(crops_dir, crop_fname)
            if crop_img.size > 0:
                cv2.imwrite(crop_path, crop_img)

            candidate_records.append({
                "patientId": pid,
                "x1": x1, "y1": y1, "x2": x2, "y2": y2,
                "confidence": float(score),
                "is_tp": is_tp,
                "best_iou": float(best_iou),
                "crop_path": crop_path
            })

    # Summary Metrics
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall = total_tp / total_gt if total_gt > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    fp_per_image = total_fp / len(test_files) if len(test_files) > 0 else 0.0

    proposals_df = pd.DataFrame(candidate_records)
    proposals_csv_path = "/kaggle/working/test_candidate_proposals.csv"
    proposals_df.to_csv(proposals_csv_path, index=False)
    print(f"\n[SAVED] Candidate proposals exported: {proposals_csv_path} ({len(proposals_df)} boxes)")

    # Print Comparison Table
    print("\n" + "=" * 75)
    print("DETECTION BENCHMARK COMPARISON TABLE")
    print("=" * 75)
    print(f"{'Metric':<22} | {'Paper Target Baseline':<22} | {'Our Achieved Baseline':<22}")
    print("-" * 75)
    print(f"{'Recall@0.5':<22} | {'80.2%':<22} | {recall*100:.1f}%")
    print(f"{'Precision@0.5':<22} | {'76.6%':<22} | {precision*100:.1f}%")
    print(f"{'F1-Score':<22} | {'78.4%':<22} | {f1*100:.1f}%")
    print(f"{'False Positives / Img':<22} | {'~0.75':<22} | {fp_per_image:.3f}")
    print("=" * 75)
else:
    print("Model weights not yet available for evaluation. Train first!")



Evaluating checkpoint on test split: /kaggle/working/experiments/baseline_rtdetr/train_run/weights/best.pt
Extracting candidate proposals from 4003 test images...

[SAVED] Candidate proposals exported: /kaggle/working/test_candidate_proposals.csv (13107 boxes)

DETECTION BENCHMARK COMPARISON TABLE
Metric                 | Paper Target Baseline  | Our Achieved Baseline 
---------------------------------------------------------------------------
Recall@0.5             | 80.2%                  | 83.0%
Precision@0.5          | 76.6%                  | 9.3%
F1-Score               | 78.4%                  | 16.8%
False Positives / Img  | ~0.75                  | 2.969
